# Assignment 5: Feature Engineering & Data Preparation — SOLVED
**Machine Learning & AI BootCamp** | *Instructor: Fahd* | Case Study: SoundWave Music Platform

## Student Information
- **Student Name:** *[Enter your name here]*
- **Batch:** *[Enter your batch here]*
- **Date:** *[Enter submission date]*

In [ ]:
# Run this cell first to import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("All required libraries successfully imported!")

## Exercise 1: Loading & Inspecting the SoundWave Dataset

In [ ]:
# TODO 1.1: Load the dataset
# The file you were given is an Excel file (.xlsx); if you have the .csv version, it is used instead.
try:
    df = pd.read_csv("soundwave_music_users.csv")
except FileNotFoundError:
    df = pd.read_excel("soundwave_music_users.xlsx")

In [ ]:
# TODO 1.2: Display shape and first 5 rows
print("Shape:", df.shape)
display(df.head())

In [ ]:
# TODO 1.3: Check churn_risk counts
print(df["churn_risk"].value_counts())
print()
print(df["churn_risk"].value_counts(normalize=True) * 100)

## Exercise 2: Datetime Feature Engineering

In [ ]:
# TODO 2.1: Convert signup_date to datetime
df["signup_date"] = pd.to_datetime(df["signup_date"])
print(df["signup_date"].dtype)

In [ ]:
# TODO 2.2: Extract signup_month
df["signup_month"] = df["signup_date"].dt.month

In [ ]:
# TODO 2.3: Extract signup_day_name
df["signup_day_name"] = df["signup_date"].dt.day_name()

In [ ]:
# TODO 2.4: Calculate tenure_days relative to 2024-01-01
df["tenure_days"] = (pd.to_datetime("2024-01-01") - df["signup_date"]).dt.days

In [ ]:
# TODO 2.5: Display first 5 rows of datetime columns
display(df[["signup_date", "signup_month", "signup_day_name", "tenure_days"]].head())

## Exercise 3: Domain-Specific Feature Creation

In [ ]:
# TODO 3.1 & 3.2: Calculate skip_rate
songs_listened = df["daily_listen_mins"] / 3.5          # estimated songs listened per day
df["skip_rate"] = df["songs_skipped_per_day"] / songs_listened

In [ ]:
# TODO 3.3: Create is_curator binary flag
df["is_curator"] = (df["playlists_created"] >= 5).astype(int)

In [ ]:
# TODO 3.4: Display the first 5 rows of the new features
display(df[["daily_listen_mins", "songs_skipped_per_day", "skip_rate", "is_curator"]].head())

## Exercise 4: Categorical Encoding

In [ ]:
# TODO 4.1: Map subscription_tier to tier_code using an ordinal dictionary
tier_map = {"Free": 0, "Bronze": 1, "Silver": 2, "Gold": 3}
df["tier_code"] = df["subscription_tier"].map(tier_map)

In [ ]:
# TODO 4.2: One-Hot encode device_type with drop_first=True
device_dummies = pd.get_dummies(df["device_type"], prefix="device_type", drop_first=True, dtype=int)
df = pd.concat([df, device_dummies], axis=1)

In [ ]:
# TODO 4.3: Display first 5 rows
display(df[["subscription_tier", "tier_code"] + list(device_dummies.columns)].head())

## Exercise 5: Discretization & Binning

In [ ]:
# TODO 5.1 & 5.2: Discretize age into age_group using pd.cut()
df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 25, 40, 70],
    labels=["Young (18-25)", "Adult (26-40)", "Mature (41+)"]
)

In [ ]:
# TODO 5.3: Display age_group value counts
print(df["age_group"].value_counts())

## Exercise 6: Stratified Train-Test Split

In [ ]:
# TODO 6.1: Define X and y
y = df["churn_risk"]
X = df[["daily_listen_mins", "songs_skipped_per_day", "tenure_days", "skip_rate", "tier_code"]]

In [ ]:
# TODO 6.2: Perform 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)

In [ ]:
# TODO 6.3: Print churn_risk percentage in y_train and y_test
print(f"Original churn rate: {y.mean()*100:.2f}%")
print(f"y_train churn rate:  {y_train.mean()*100:.2f}%")
print(f"y_test churn rate:   {y_test.mean()*100:.2f}%")

## Exercise 7: Leakage-Free Feature Scaling

In [ ]:
# TODO 7.1: Initialize StandardScaler
scaler = StandardScaler()

In [ ]:
# TODO 7.2: Fit on X_train, transform X_train and X_test
X_train_scaled = scaler.fit_transform(X_train)   # fit ONLY on training data
X_test_scaled = scaler.transform(X_test)         # reuse train mean/std - no refitting

In [ ]:
# TODO 7.3: Verify mean and standard deviation of X_train_scaled
print("Mean of each column (should be ~0):", np.round(X_train_scaled.mean(axis=0), 4))
print("Std of each column  (should be ~1):", np.round(X_train_scaled.std(axis=0), 4))

## Exercise 8: Scikit-Learn Preprocessing Pipeline (Synthesis)

In [ ]:
# TODO 8.1: Define feature subsets
num_features = ['daily_listen_mins', 'songs_skipped_per_day', 'tenure_days']
cat_features = ['primary_genre', 'device_type']

In [ ]:
# TODO 8.2: Build ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_features)
])

In [ ]:
# TODO 8.3: Build Pipeline with LogisticRegression
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42))
])

In [ ]:
# TODO 8.4: Split features and target
X_pipe = df[num_features + cat_features]
y_pipe = df["churn_risk"]

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pipe, y_pipe, test_size=0.20, random_state=42, stratify=df["churn_risk"]
)

In [ ]:
# TODO 8.5: Fit pipeline and print test accuracy
model_pipeline.fit(X_train_p, y_train_p)
test_preds = model_pipeline.predict(X_test_p)
print(f"Test Accuracy: {accuracy_score(y_test_p, test_preds):.3f}")

### 🎉 Assignment 5 complete!